# FineTunePilot — Stage 1: baseline QLoRA training on Kaggle

Fine-tunes `Qwen/Qwen2.5-0.5B-Instruct` with QLoRA on the 60-row synthetic tool-calling set and
writes a run log to `runs/<run_id>.json`.

**Before you start:** Settings → Accelerator → **GPU T4** (x1 or x2, only one is used),
Settings → **Internet on**. The repo must be public (or clone it with a token from Kaggle Secrets).

Expected: about 40 optimizer steps, a few minutes on a T4. Check that the loss goes down.

## 1. Check the GPU
You should see a Tesla T4 with about 15 GB.

In [ ]:
!nvidia-smi

## 2. Get the code and install it
Clones branch `claude/zen-meitner-hetiul` and installs it with the `gpu` extra (bitsandbytes, accelerate). Kaggle's own CUDA build of torch is kept.

In [ ]:
%cd /kaggle/working
!rm -rf Finetune-pilot-ai
!git clone --depth 1 --branch claude/zen-meitner-hetiul https://github.com/manojastro/Finetune-pilot-ai.git
%cd Finetune-pilot-ai
!pip install -q -e ".[gpu]"

## 3. Check library versions
The code needs transformers 5.x and trl 1.x. `cuda` must be `True`.

In [ ]:
!python -c "import torch, transformers, trl, peft, bitsandbytes; print('torch', torch.__version__, 'cuda', torch.cuda.is_available()); print('transformers', transformers.__version__, 'trl', trl.__version__, 'peft', peft.__version__, 'bnb', bitsandbytes.__version__)"

## 4. Use one GPU only
On "T4 x2" the HF Trainer would otherwise split batches across both GPUs, which changes batch size and memory numbers.

In [ ]:
%env CUDA_VISIBLE_DEVICES=0

## 5. Quick dry run (optional, ~10 s)
Tiny random model on CPU, 2 steps. Proves the install works before using GPU time.

In [ ]:
!finetunepilot train --config configs/qwen05b_toolcall.yaml --dry-run --output-dir /tmp/dryrun

## 6. Real training
QLoRA: 4-bit NF4 base model, LoRA r=16 on all linear layers, fp16. The adapter is saved to `outputs/qwen05b_toolcall/<run_id>/`.

In [ ]:
!finetunepilot train --config configs/qwen05b_toolcall.yaml

## 7. Show the run log
Prints the newest real (non dry-run) run JSON. Copy this whole output back into Claude Code with the paste-back prompt.

In [ ]:
import glob, json, os
paths = sorted(p for p in glob.glob("runs/*.json") if "-dryrun-" not in p)
log = json.load(open(paths[-1]))
print(json.dumps(log, indent=2))

## 8. Did the loss go down?
First vs last logged loss, and the saved adapter files.

In [ ]:
hist = log["attempts"][-1]["result"].get("loss_history", []) if log["status"] == "success" else []
if hist:
    print(f"first loss {hist[0]['loss']:.4f} (step {hist[0]['step']:.0f}) -> last loss {hist[-1]['loss']:.4f} (step {hist[-1]['step']:.0f})")
    print("adapter files:", sorted(os.listdir(log["artifacts"]["adapter_dir"])))
else:
    print("Run failed:", log["errors"][:1])